In [24]:
import logging
import os
import time

import tensorflow as tf

from migration import datasets as datasets
from migration.elbo import elbo
from migration.get_config import config
from migration.models import vrnn


# get batch and model
def create_dataset_and_model(config, shuffle, repeat):

    inputs, targets, mmsis, time_starts, time_ends, lengths, mean =  datasets.create_AIS_dataset('../../data/ct_2017010203_10_20/ct_2017010203_10_20_train.pkl', 
                    '../../data/ct_2017010203_10_20/mean.pkl',
                    32, # batch size
                    99999, # not used lol
                    300,
                    300, 
                    30,
                    72, 
                    shuffle=False,
                    repeat=False)
    # Convert the mean of the training set to logit space so it can be used to
    # initialize the bias of the generative distribution.
    generative_bias_init = -tf.math.log(1. / tf.clip_by_value(mean, 0.0001, 0.9999) - 1)
    generative_distribution_class = vrnn.ConditionalBernoulliDistribution
    model = vrnn.create_vrnn(inputs.get_shape().as_list()[2],
                             config.latent_size,
                             generative_distribution_class,
                             generative_bias_init=generative_bias_init,
                             raw_sigma_bias=0.5)
    return inputs, targets, mmsis, time_starts, time_ends, lengths, model


# def restore_checkpoint_if_exists(saver, sess, logdir):
#     """Looks for a checkpoint and restores the session from it if found.
#     Args:
#       saver: A tf.train.Saver for restoring the session.
#       sess: A TensorFlow session.
#       logdir: The directory to look for checkpoints in.
#     Returns:
#       True if a checkpoint was found and restored, False otherwise.
#     """
#     checkpoint = tf.train.get_checkpoint_state(logdir)
#     if checkpoint:
#         checkpoint_name = os.path.basename(checkpoint.model_checkpoint_path)
#         full_checkpoint_path = os.path.join(logdir, checkpoint_name)
#         saver.restore(sess, full_checkpoint_path)
#         return True
#     return False


# def wait_for_checkpoint(saver, sess, logdir):
#     while True:
#         if restore_checkpoint_if_exists(saver, sess, logdir):
#             break
#         else:
#             tf.compat.v1.logging.info("Checkpoint not found in %s, sleeping for 60 seconds."
#                       % logdir)
#             time.sleep(60)


def run_train(config):

    def create_logging_hook(step, bound_value):
        """Creates a logging hook that prints the bound value periodically."""
        bound_label = config.bound + " bound"
        if config.normalize_by_seq_len:
            bound_label += " per timestep"
        else:
            bound_label += " per sequence"
        def summary_formatter(log_dict):
            return "Step %d, %s: %f" % (
                        log_dict["step"], bound_label, log_dict["bound_value"])
        logging_hook = tf.estimator.LoggingTensorHook(
                                {"step": step,
                                 "bound_value": bound_value},
                                every_n_iter=1,
                                formatter=summary_formatter)
        return logging_hook

    def create_loss():
        """Creates the loss to be optimized.
        Returns:
            bound: A float Tensor containing the value of the bound that is
                   being optimized.
            loss: A float Tensor that when differentiated yields the gradients
                to apply to the model. Should be optimized via gradient descent.
        """
        inputs, targets, _, _, _, lengths, model = create_dataset_and_model(config,
                                                               shuffle=True,
                                                               repeat=True)
        
        ll_per_seq, _, _, _ = elbo(model, (inputs, targets),
                                              lengths,
                                              num_samples=1)
        # Compute loss scaled by number of timesteps.
        ll_per_t = tf.reduce_mean(input_tensor=ll_per_seq / tf.cast(lengths, dtype=tf.float32))
        ll_per_seq = tf.reduce_mean(input_tensor=ll_per_seq)

        tf.compat.v1.summary.scalar("train_ll_per_seq", ll_per_seq)
        tf.compat.v1.summary.scalar("train_ll_per_t", ll_per_t)

        if config.normalize_by_seq_len:
            return ll_per_t, -ll_per_t
        else:
            return ll_per_seq, -ll_per_seq

    def create_graph():
        """Creates the training graph."""
        global_step = tf.compat.v1.train.get_or_create_global_step()
        bound, loss = create_loss()
        opt = tf.compat.v1.train.AdamOptimizer(config.learning_rate)
        grads = opt.compute_gradients(loss, var_list=tf.compat.v1.trainable_variables())
        train_op = opt.apply_gradients(grads, global_step=global_step)
        return bound, train_op, global_step

    device = tf.compat.v1.train.replica_device_setter(ps_tasks=config.ps_tasks)
    with tf.Graph().as_default():
        tf.compat.v1.set_random_seed(1111)
        with tf.device(device):
            bound, train_op, global_step = create_graph()
            log_hook = create_logging_hook(global_step, bound)
            start_training = not config.stagger_workers
            with tf.compat.v1.train.MonitoredTrainingSession(master=config.master,
                                                   is_chief=config.task == 0,
                                                   hooks=[log_hook],
                                                #    checkpoint_dir=config.logdir,
                                                   save_checkpoint_secs=120,
                                                   save_summaries_steps=1,
                                                   log_step_count_steps=1) as sess:
                cur_step = -1
                for _ in range(5):
                    # if config.task > 0 and not start_training:
                    #     cur_step = sess.run(global_step)
                    #     tf.compat.v1.logging.info("task %d not active yet, sleeping at step %d" %
                    #         (config.task, cur_step))
                    #     time.sleep(30)
                    #     if cur_step >= config.task * 1000:
                    #         start_training = True
                    # else:
                    _, cur_step = sess.run([train_op, global_step])
    #                         _, cur_step = sess.run([train_op, global_step])




In [26]:
print(config.trainingset_path)
fh = logging.FileHandler(os.path.join(config.logdir,config.log_filename+".log"))
tf.compat.v1.logging.set_verbosity(tf.compat.v1.logging.INFO)
# get TF logger
logger = logging.getLogger('tensorflow')
logger.addHandler(fh)
run_train(config)  

./data/ct_2017010203_10_20/ct_2017010203_10_20_train.pkl
INFO:tensorflow:Graph was finalized.
INFO:tensorflow:Running local_init_op.
INFO:tensorflow:Done running local_init_op.
INFO:tensorflow:Step 1, elbo bound per timestep: -21.556837
INFO:tensorflow:Step 2, elbo bound per timestep: -21.373081
INFO:tensorflow:Step 3, elbo bound per timestep: -21.212431
INFO:tensorflow:Step 4, elbo bound per timestep: -21.452400
INFO:tensorflow:Step 5, elbo bound per timestep: -21.973835
